In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
%pip install ../databricks_helpers
%pip install meteostat
dbutils.library.restartPython()

In [0]:
from databricks_helpers.databricks_helper import DatabricksHelpers
dbh = DatabricksHelpers(dbutils, "aviation_analytics",spark)

In [0]:
%sql
--CREATE VOLUME IF NOT EXISTS raw_data;
SELECT current_catalog(), current_schema();

In [0]:
#IATA_CODE to LAT, LONG taken from https://ourairports.com/data/
df=spark.read.csv(f"{dbh.volume_directory()}/airports.csv", header=True, inferSchema=True)


In [0]:
df.filter(df.iata_code.isNotNull())\
.withColumnsRenamed({"longitude_deg":"long","iata_code":"iata","latitude_deg":"lat"})\
.drop("id","iso_region","municipality","scheduled_service","gps_code","local_code","home_link","wikipedia_link","keywords")\
.write.mode("overwrite").saveAsTable("airports")

In [0]:
%sql
CREATE VIEW if not exists req_airports_view as select * From airports where iata in (select distinct origin from bronze_flight_data);

In [0]:
from datetime import datetime
import meteostat as ms


meteo_daily_data=f"{dbh.volume_directory()}/daily_meteo_data"
start = datetime(2022, 1, 1)
end = datetime(2025, 12, 31)
weather_list = []
required_locations = [(row.iata, (row.lat, row.long)) for row in spark.table("req_airports_view").select("iata","lat","long").collect()]

In [0]:
import time
import pickle
import os

monthly_limit = 500
requests_made = 0
batch_size = 3
checkpoint_file = f"{dbh.volume_directory()}/weather_checkpoint.pkl"
# Load checkpoint if exists, so that if any batch fails it can start from there
#checkpointing in case there is a failure
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "rb") as f:
        checkpoint = pickle.load(f)
        start_idx = checkpoint.get("start_idx", 0)
        requests_made = checkpoint.get("requests_made", 0)
        weather_list = checkpoint.get("weather_list", [])
else:
    start_idx = 0

for i in range(start_idx, min(len(required_locations), monthly_limit), batch_size): #iteracting batch of 3 so as to abide the rate limit of 3 req/second
    batch = required_locations[i:i+batch_size]
    for code, (lat, lon) in batch:
        if requests_made >= monthly_limit:
            break
        point = ms.Point(lat, lon)
        data = ms.Daily(point, start, end).fetch().reset_index()
        data['iata_code'] = code
        weather_list.append(data)
        requests_made += 1
    # Save checkpoint after each batch
    with open(checkpoint_file, "wb") as f:
        pickle.dump({
            "start_idx": i + batch_size,
            "requests_made": requests_made,
            "weather_list": weather_list
        }, f)
    time.sleep(1)  # Enforce 3 requests per second

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import meteostat as ms
import pandas as pd
import time

spark = SparkSession.builder.getOrCreate()

monthly_limit = 500
batch_size = 3

# Convert input list to Spark DataFrame
df = spark.createDataFrame(required_locations, ["iata_code", "coords"])


In [0]:
#Split to lat long
df = df.withColumn("lat", col("coords._1")) \
       .withColumn("lon", col("coords._2")).drop("coords")


In [0]:

# Limit total requests
df = df.limit(monthly_limit)

# Repartition to control parallelism
df = df.repartition(2)  # tune this based on cluster size

def fetch_weather_partition(iterator):
    import meteostat as ms
    import pandas as pd
    import time

    results = []
    requests_made = 0

    batch = []

    for row in iterator:
        batch.append(row)

        if len(batch) == batch_size:
            for r in batch:
                try:
                    point = ms.Point(r.lat, r.lon)
                    station = ms.stations.nearby(point, limit=1)
                    data = ms.daily(station, start, end).fetch().reset_index()
                    data['iata_code'] = r.iata_code
                    results.append(data)
                    requests_made += 1
                except Exception as e:
                    print(f"Error: {e}")

            batch = []
            time.sleep(1)  # rate limit (3 req/sec)

    # leftover batch, if batch not multiple of 4
    for r in batch:
        try:
            point = ms.Point(r.lat, r.lon)
            station = ms.stations.nearby(point, limit=1)
            data = ms.daily(station, start, end).fetch().reset_index()
            data['iata_code'] = r.iata_code
            results.append(data)
        except Exception as e:
            print(f"Error: {e}")

    if results:
        return iter([pd.concat(results)]) #this one has internal fields of pandas like _flags, _references which cant be converted directly.
        #return iter(final_df.to_dict(orient="records")) 
    else:
        return iter([])

# Apply distributed processing
weather_rdd = df.rdd.mapPartitions(fetch_weather_partition)

# Convert back to Spark DataFrame
#weather_df = spark.createDataFrame(weather_rdd) doesnt work with pd.condat and returning pandas dataframe



In [0]:
pdf_lits=weather_rdd.collect()

In [0]:
final_pdf = pd.concat(pdf_lits, ignore_index=True)

In [0]:
#saving it in json
final_pdf['time'] = final_pdf['time'].dt.strftime('%Y-%m-%d')
final_pdf.to_json(f"{dbh.volume_directory()}/weather_data.json", orient='records', lines=True)

In [0]:
# Step 2: Fix unsupported types
final_pdf = final_pdf.astype({
    col: "float64"
    for col in final_pdf.select_dtypes(include=["uint8", "uint16", "uint32", "uint64"]).columns
})

In [0]:
#was getting error
# /databricks/spark/python/pyspark/sql/pandas/conversion.py:473: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
#   Cannot convert pyarrow.lib.ChunkedArray to pyarrow.lib.Array
# Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
#   warn(msg)
#Getting this array on arrow backend type
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false") 
weather_df = spark.createDataFrame(final_pdf)

In [0]:
display(weather_df)

In [0]:
# final_weather_df['year'] = pd.to_datetime(weather_df['time']).dt.year
# final_weather_df['month'] = pd.to_datetime(weather_df['time']).dt.month
# final_weather_df['day'] = pd.to_datetime(weather_df['time']).dt.day
from pyspark.sql.functions import year,month,dayofmonth

final_weather_df = weather_df.withColumn("year", year("time"))\
 .withColumn("month", month("time"))\
 .withColumn("day", dayofmonth("time"))

final_weather_df.write.mode("overwrite").partitionBy("year", "month", "day").parquet(f"{dbh.volume_directory()}/weather_data")